# Stage One: Data Inventory and Reliability Assessment

This notebook is a record of the code for Stage One, written so that the steps can be followed in order. What each result showed, and the reasoning behind each decision, are set out in `md/01_data_inventory_writeup.md`.

---

## 1. Retrieving the dataset

The code below downloads the dataset from Kaggle and prints the folder it was saved to along with the files it contains. Fetching the data in code rather than by hand means that anyone running the notebook gets the same file from the same place, with no dependency on a copy held on one machine.

The full version of the dataset is used. The shorter version in wider circulation leaves out the customer lifetime value figure, the recorded reason for departure and the location fields, all of which this project needs.

In [1]:
import kagglehub, os

path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")
print("Path:", path)
print(os.listdir(path))

c:\Users\IC Clearwater\OneDrive\Documents\GitHub\Customer_Lifetime_Value_Churn_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path: C:\Users\IC Clearwater\.cache\kagglehub\datasets\yeanzc\telco-customer-churn-ibm-dataset\versions\1
['Telco_customer_churn.xlsx']


## 2. Confirming the structure of the dataset

This loads the spreadsheet into a pandas table, then prints the number of rows and columns, the full list of field names, and the first ten records.

The field names are checked before any other work is done. The project depends on particular fields being present, and if one were missing that needs to be known at the start rather than after the work is built on top of it. The first ten records show how the values are formatted.

In [2]:
import pandas as pd

df = pd.read_excel(f"{path}/Telco_customer_churn.xlsx")
print(df.shape)
print(df.columns.tolist())
pd.set_option('display.max_columns', None)
df.head(10)

(7043, 33)
['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices
5,4190-MFLUW,1,United States,California,Los Angeles,90020,"34.066367, -118.309868",34.066367,-118.309868,Female,No,Yes,No,10,Yes,No,DSL,No,No,Yes,Yes,No,No,Month-to-month,No,Credit card (automatic),55.20,528.35,Yes,1,78,5925,Competitor offered higher download speeds
6,8779-QRDMV,1,United States,California,Los Angeles,90022,"34.02381, -118.156582",34.023810,-118.156582,Male,Yes,No,No,1,No,No phone service,DSL,No,No,Yes,No,No,Yes,Month-to-month,Yes,Electronic check,39.65,39.65,Yes,1,100,5433,Competitor offered more data
7,1066-JKSGK,1,United States,California,Los Angeles,90024,"34.066303, -118.435479",34.066303,-118.435479,Male,No,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes,1,92,4832,Competitor made better offer
8,6467-CHFZW,1,United States,California,Los Angeles,90028,"34.099869, -118.326843",34.099869,-118.326843,Male,No,Yes,Yes,47,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.35,4749.15,Yes,1,77,5789,Competitor had better devices
9,8665-UTDHZ,1,United States,California,Los Angeles,90029,"34.089953, -118.294824",34.089953,-118.294824,Male,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,No,Electronic check,30.20,30.2,Yes,1,97,2915,Competitor had better devices


## 3. Reviewing field types and missing values

Two checks run here. The first prints how each field is stored, as a number, a date or as text. The second counts missing values, limited to the fields where any occur.

Field types matter for a specific reason. A field holding money has to be stored as a number before it can be added up or averaged. Where a money field is stored as text, at least one entry in it cannot be read as a number, and that needs looking into.

In [3]:
pd.set_option('display.max_rows', 40)
print(df.dtypes)
print("\nNulls:")
print(df.isna().sum()[df.isna().sum() > 0])

CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object

Nulls:
Churn Reason    5174


## 4. Completing the field type inventory

The previous output was cut short on display, so the last ten field types are printed on their own.

Every field is inspected. A field that was never looked at is an assumption rather than a fact.

In [4]:
print(df.dtypes.tail(10))

Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object


## 5. Investigating the non-numeric entries

This tests each value in the total charges field to find the ones that cannot be read as a number, counts them, and then shows those records alongside tenure, monthly charge and departure status.

The field is not converted at this point. Converting first and investigating afterwards would overwrite the unreadable entries and destroy the evidence needed to work out why they are there. Showing the affected records next to tenure and departure status is what makes it possible to see what those customers have in common.

In [5]:
bad = pd.to_numeric(df['Total Charges'], errors='coerce').isna()
print(bad.sum())
print(df.loc[bad, ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Label']])

11
      Tenure Months  Monthly Charges Total Charges Churn Label
2234              0            52.55                        No
2438              0            20.25                        No
2568              0            80.85                        No
2667              0            25.75                        No
2856              0            56.05                        No
4331              0            19.85                        No
4687              0            25.35                        No
5104              0            20.00                        No
5719              0            19.70                        No
6772              0            73.35                        No
6840              0            61.90                        No


## 6. Correcting the field and summarising the numeric data

This converts total charges to a number, puts zero in place of the entries that could not be read, and confirms that the field type has changed. It then prints summary statistics for the main numeric fields.

Summary statistics are the count, average, spread, smallest and largest values, and the quarter points of each field. Producing them for every numeric field at once gives the range and shape of each one in a single view, which is what makes values that cannot be right easy to notice.

In [6]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce').fillna(0)
print(df['Total Charges'].dtype)
print(df[['Tenure Months','Monthly Charges','Total Charges','CLTV']].describe())

float64
       Tenure Months  Monthly Charges  Total Charges         CLTV
count    7043.000000      7043.000000    7043.000000  7043.000000
mean       32.371149        64.761692    2279.734304  4400.295755
std        24.559481        30.090047    2266.794470  1183.057152
min         0.000000        18.250000       0.000000  2003.000000
25%         9.000000        35.500000     398.550000  3469.000000
50%        29.000000        70.350000    1394.550000  4527.000000
75%        55.000000        89.850000    3786.600000  5380.500000
max        72.000000       118.750000    8684.800000  6500.000000


## 7. Testing the reliability of the supplied lifetime value figure

This measures rank correlation between the supplied lifetime value figure and four other fields: monthly charge, total charges, tenure and departure status.

Whatever formula is used behind it, a lifetime value figure is built from what a customer pays and how long the relationship lasts, so a sound figure has to move with both. Rank correlation is used rather than a measure of size. It asks only whether two fields put customers in the same order, ignoring the scale each is measured on, which suits a figure already suspected of using a scale of its own. The result runs between minus one and one, where a figure near one means the two order customers almost identically and a figure near zero means no relationship at all.

In [7]:
print(df[['CLTV','Monthly Charges','Total Charges','Tenure Months','Churn Value']].corr(method='spearman')['CLTV'])

CLTV               1.000000
Monthly Charges    0.107944
Total Charges      0.310721
Tenure Months      0.367039
Churn Value       -0.123627
Name: CLTV, dtype: float64


---

Stage One ends here. The findings, including the decision to reject the supplied lifetime value figure and the treatment applied to the eleven unbilled records, are set out in `md/01_data_inventory_writeup.md`.

**Next stage:** build the four linked tables, derive the measures the model needs, and calculate customer value in SQL.